# TN2 — Tầm nhìn DS-TCN 64 kênh, VÒNG XÁC NHẬN 4 fold

## Vòng sàng lọc cho kết quả gì

Vòng sàng lọc một fold, một seed. Tầm nhìn càng dài điểm càng tệ. Ba cấu hình
notebook này chạy tiếp:

| kernel | tầm nhìn | điểm `val_KL` | train_mse | train_pearson |
|---:|---:|---:|---:|---:|
| **5** | **121** | **0,8458** | 0,02127 | 0,5851 |
| **7** | **181** | **0,8176** | 0,02059 | 0,5921 |
| **9** | **241** | **0,8041** | 0,01939 | 0,6041 |

Tương quan tầm nhìn với điểm: **−0,983**.

Chỗ đáng chú ý: cùng lúc điểm tụt thì model **dự báo giỏi lên** — `train_mse`
giảm 13%, `train_pearson` tăng. Model càng giỏi dự báo càng chọn kênh dở.

Cơ chế có thể giải thích: tiêu chí chọn kênh là "ứng viên nào tự dự báo được
chính nó tốt nhất". Model tầm nhìn ngắn chỉ đoán giỏi sóng **thật sự tuần
hoàn**; model tầm nhìn dài đoán giỏi **mọi sóng trơn**, kể cả kênh nhiễu có cấu
trúc. Tầm nhìn ngắn hoạt động như một bộ lọc — dở đúng chỗ cần dở.

Đây mới là **giả thuyết**, chưa chứng minh.

## Vòng này chạy gì

Bỏ hai cấu hình tầm nhìn dài nhất (301, 361), giữ **ba cấu hình đầu** và chạy
**đủ bốn fold**, một seed.

| kernel | tầm nhìn | tham số | `val_KL` đã có |
|---:|---:|---:|---:|
| 5 | 121 | 38.105 | 0,8458 |
| 7 | 181 | 39.129 | 0,8176 |
| 9 | 241 | 40.153 | 0,8041 |

**Fold `val_KL` của cả ba đã chạy ở vòng sàng lọc.** Ô khôi phục kéo chúng về
từ Drive, nên `run_cv.py` sẽ in `đã có kết quả — bỏ qua` và chỉ train ba fold
còn lại. Ước lượng **1,5–2 giờ** thay vì gấp bốn.

Chạy xong, dòng `TONG` tự được ghi và `compare_cv` hiện `cv_score` đầy đủ.

## Vì sao vẫn chưa kết luận được sau vòng này

Một seed. `seed_std` của tám cấu hình TN1 trải từ 0,0007 tới 0,0108, mỗi kiến
trúc một khác — không mượn của nhau được. Muốn nói ba cấu hình khác nhau thật
thì phải chạy đủ ba seed cho chính chúng.

Vòng này trả lời câu hẹp hơn: **xu hướng thấy ở một fold có giữ nguyên trên đủ
bốn fold không?** Nếu `val_DF` — fold khó nhất — cho thứ tự ngược lại thì kết
luận từ `val_KL` là do fold chứ không do tầm nhìn.

## Mốc để đặt cạnh

| | tham số | tầm nhìn | cv_score |
|---|---:|---:|---:|
| DS-TCN-64 k3n4 no_norm do0.2 | 37.081 | **61** | 0,760878 ± 0,003095 *(3 seed)* |
| LSTM-352 | 1.502.713 | — | 0,756992 ± 0,004156 |
| DS-TCN-64 (TN1 gốc) | 56.281 | 253 | 0,742117 ± 0,000677 |

Dòng đầu là cùng kiến trúc, kernel 3, tầm nhìn 61 — điểm nối đầu bảng của thang
này.

## 1. Chuẩn bị Colab

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Tải mã nguồn.

In [ ]:
!rm -rf /content/UWB_RADAR
!git clone -q https://github.com/quangminhho004-blip/UWB_RADAR.git /content/UWB_RADAR
%cd /content/UWB_RADAR
!python scripts/setup_colab.py

Lấy `by_user/` và `windows/` từ Drive.

In [ ]:
!python scripts/restore_processed_data_on_drive.py

Khôi phục fold `val_KL` đã chạy ở vòng sàng lọc.

**Bước này quyết định notebook chạy 1,5 giờ hay 6 giờ.** Không có nó thì
`run_cv.py` train lại cả bốn fold. Phải thấy in ra ba dòng `giải nén`.

In [ ]:
# Khôi phục kết quả đã chạy trước khi phiên bị ngắt.
#
# Mỗi tệp nén chứa một bản runs/<thực nghiệm>/summary.csv của riêng nó. Giải
# hết vào cùng một thư mục runs/ thì tệp giải sau ĐÈ summary.csv của tệp trước.
# Mà sorted() xếp "..._a0_corr0.9..." đứng SAU "..._a0.9_corr0.9...", vì trong
# bảng mã ký tự "_" lớn hơn "." — nên bản ít dòng nhất lại là bản đè cuối cùng.
# Sửa: mỗi tệp nén giải vào một thư mục tạm riêng, gộp mọi dòng lại rồi mới ghi
# runs/summary.csv một lần. Thứ tự giải nén không còn ảnh hưởng gì nữa.
import csv, glob, os, shutil, subprocess, tempfile

MAU_ZIP = "/content/drive/MyDrive/mobivital/tn2_rf_*c64*.zip"

rows, seen = [], set()

def collect(summary_path):
    for r in csv.DictReader(open(summary_path)):
        key = (r.get("experiment"), r.get("run_id"))
        if key not in seen:
            seen.add(key)
            rows.append(r)

for f in sorted(glob.glob(MAU_ZIP)):
    tmp = tempfile.mkdtemp()
    subprocess.run(["unzip", "-oq", f, "-d", tmp], check=True)
    for s in glob.glob(tmp + "/*/summary.csv"):
        collect(s)
        os.remove(s)   # gộp xong thì bỏ, để bước chép dưới không đè nhau nữa
    # Checkpoint và curve.csv nằm trong thư mục riêng của từng lần chạy, tên
    # không trùng nhau, nên chép chồng lên runs/ là an toàn.
    for d in os.listdir(tmp):
        shutil.copytree(tmp + "/" + d, "runs/" + d, dirs_exist_ok=True)
    shutil.rmtree(tmp)

# Dòng đã sinh ra trong chính phiên này cũng phải giữ lại.
if os.path.exists("runs/summary.csv"):
    collect("runs/summary.csv")

if rows:
    # run_cv.py tra runs/summary.csv, còn tệp nén chỉ có runs/<thực nghiệm>/summary.csv
    cols = []
    for r in rows:
        for k in r:
            if k not in cols:
                cols.append(k)
    with open("runs/summary.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=cols, restval="")
        w.writeheader()
        w.writerows(rows)
print("khôi phục", len(rows), "dòng vào runs/summary.csv")

## 2. Kiểm ba bản cài đặt

**Đọc dòng cuối mỗi lệnh.** Phải là `TẤT CẢ ĐẠT`.

In [ ]:
!python scripts/check_model.py --model ds_tcn --channels 64 \
    --kernel_size 5 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element

In [ ]:
!python scripts/check_model.py --model ds_tcn --channels 64 \
    --kernel_size 7 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element

In [ ]:
!python scripts/check_model.py --model ds_tcn --channels 64 \
    --kernel_size 9 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element

## 3. Chạy đủ 4 fold, một seed

Không có `--folds` nên chạy đủ bốn. Fold `val_KL` đã có sẽ bị bỏ qua — tìm dòng
`đã có kết quả 0.xxxx — bỏ qua, không train lại` để chắc ô khôi phục đã ăn.

Mỗi cấu hình khoảng **30 phút** cho ba fold còn lại.

**kernel 5 — tầm nhìn 121, 38.105 tham số, `val_KL` đã có 0,8458**

In [ ]:
!python scripts/run_cv.py --experiment tn2_rf --model ds_tcn --channels 64 \
    --kernel_size 5 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element --seed 0

**kernel 7 — tầm nhìn 181, 39.129 tham số, `val_KL` đã có 0,8176**

In [ ]:
!python scripts/run_cv.py --experiment tn2_rf --model ds_tcn --channels 64 \
    --kernel_size 7 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element --seed 0

**kernel 9 — tầm nhìn 241, 40.153 tham số, `val_KL` đã có 0,8041**

In [ ]:
!python scripts/run_cv.py --experiment tn2_rf --model ds_tcn --channels 64 \
    --kernel_size 9 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element --seed 0

## 4. Cất kết quả

In [ ]:
!python scripts/save_results.py tn2_rf --out tn2_rf_c64_4fold

## 5. Bảng so

Giờ đã đủ bốn fold nên dòng `TONG` có, `compare_cv` hiện `cv_score` thật.

In [ ]:
!python scripts/compare_cv.py --experiment tn2_rf

## 6. Xu hướng còn giữ không

In điểm từng fold để xem thứ tự ba cấu hình có giống nhau ở mọi fold không. Nếu
`val_DF` — fold khó nhất — cho thứ tự ngược thì kết luận từ `val_KL` là do fold
chứ không do tầm nhìn.

In [ ]:
import csv, re
RF = {5: 121, 7: 181, 9: 241}
rows = [r for r in csv.DictReader(open("runs/tn2_rf/summary.csv"))
        if "_c64_" in r["run_id"] and r["fold"] != "TONG"]
for r in sorted(rows, key=lambda r: (r["fold"], r["run_id"])):
    k = int(re.search(r"_k(\d+)_", r["run_id"]).group(1))
    print(" ", r["fold"], " kernel", k, " tầm nhìn", RF[k], " ", r["score_macro"])

## 7. Ngắt phiên

In [ ]:
from google.colab import runtime
runtime.unassign()